# COVER-KBC — Post-Architecture Real-Model Runtime Smoke

Proves the architecture that passed under `ScriptedRuntime` also executes
against the two **real frozen models**.

This notebook answers runtime questions only — can the weights load, do the
native tokenizer paths work, does `score_labels` produce usable logits, does
Module 17's live call plan cost what Module 20 says, is the shadow stack
isolated from production output.

**It reads no benchmark data, has no TRAIN/VAL/TEST switch, and computes no
accuracy.** A factually wrong answer is still a runtime PASS if every contract
executed correctly.

Module 20 and Module 21 stay **disabled**: they have no TRAIN calibration, and
a synthetic package would not prove production readiness.


## 1. Environment, exact commit, budget and model pins


In [ ]:
# ============================================================
# COVER-KBC — Post-Architecture Real-Model Runtime Smoke
# EDIT THESE, then Runtime > Run all.
# ============================================================
REPO_URL   = "https://github.com/vquclinh/FactElicit-AKBC.git"
REPO_SHA   = ""        # <- REQUIRED: exact committed SHA to audit
REPO_ROOT  = "/content/FactElicit-AKBC"
CACHE_ROOT = "/content/hf-cache"      # model/weight cache
OUTPUT_ROOT= "/content/smoke-out"
CONFIG     = "configs/experiments/cover_kbc_v2_mistral24_qwen4.yaml"

# This notebook runs NO benchmark split. There is no TRAIN, VAL or TEST switch,
# and it computes no accuracy. It answers runtime questions only.

import os, subprocess, sys, json, pathlib, shutil
os.makedirs(CACHE_ROOT, exist_ok=True); os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["TRANSFORMERS_CACHE"] = CACHE_ROOT

if not pathlib.Path(REPO_ROOT).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)
os.chdir(REPO_ROOT)

assert REPO_SHA, "set REPO_SHA to the exact commit this smoke audits"
subprocess.run(["git", "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "checkout", "--force", REPO_SHA], check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                      text=True, check=True).stdout.strip()
assert head == REPO_SHA, f"HEAD {head} != requested {REPO_SHA}"
print("repo HEAD verified:", head)

# The benchmark snapshot must be untouched before anything else runs.
for args in (["git","status","--porcelain","benchmark/"],
             ["git","diff","--","benchmark/"],
             ["git","diff","--cached","--","benchmark/"]):
    out = subprocess.run(args, capture_output=True, text=True, check=True).stdout
    assert out == "", f"benchmark/ is dirty: {args}\n{out}"
print("benchmark integrity: clean")

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", ".[hf]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "bitsandbytes",
                "mistral-common"], check=True)

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  {p.total_memory/1e9:.1f} GB")
print("disk free:", f"{shutil.disk_usage('/content').free/1e9:.1f} GB")

# The 32B parameter budget must pass before any weight is fetched.
subprocess.run([sys.executable, "scripts/audit_model_budget.py", CONFIG], check=True)

import yaml
profile = yaml.safe_load(open(CONFIG))["model_profile"]
for role in ("enumerator", "verifier"):
    spec = profile[role]
    print(f"{role}: {spec['model_id']} @ {spec['revision']} "
          f"tokenizer={spec['tokenizer_backend']} quant={spec.get('quantization')}")


## 2. Run the smoke and print the machine-readable summary


In [ ]:
# ============================================================
# Run the smoke. Loads the two real frozen models, runs the primitive
# and composed checks, and writes one machine-readable summary.
#
# Expect this to take a while: Mistral-Small-24B must download and load.
# ============================================================
import subprocess, sys, json, pathlib

OUT = pathlib.Path(OUTPUT_ROOT) / "real_model_architecture_smoke_summary.json"
LOG = pathlib.Path(OUTPUT_ROOT) / "real_model_smoke.log"

with LOG.open("w") as log:
    process = subprocess.Popen(
        [sys.executable, "scripts/real_model_smoke.py",
         "--config", CONFIG, "--out", str(OUT)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end=""); log.write(line)
    code = process.wait()

summary = json.loads(OUT.read_text())
print("\n" + "=" * 64)
print("RESULT:", summary["result"])
print("repo_sha:", summary["repo_sha"])
print("enumerator:", summary["models"]["enumerator"])
print("verifier:  ", summary["models"]["verifier"])
if summary["result"] == "PASS":
    m17 = summary["m17_call_plan"]
    print(f"M17 cold expected={m17['expected_cold_calls']} "
          f"observed={m17['observed_cold_calls']}  |  "
          f"warm expected={m17['expected_warm_calls']} "
          f"observed={m17['observed_warm_calls']}")
    c = summary["composed"]
    print(f"calls: core={c['production_core_calls']} "
          f"upgraded={c['upgraded_shadow_calls']} "
          f"shadow-only={c['shadow_only_calls']}")
    print("production output unchanged:", c["production_output_unchanged"])
    print("M7 budget unchanged:", c["m7_budget_unchanged"])
    print("specialist families:", c["specialist_families"])
    print("M18 mechanisms executed:", c["m18_mechanisms_executed"])
else:
    for err in summary["errors"]:
        print(err["type"], ":", err["message"])
print("=" * 64)
print("summary:", OUT)
print("log:    ", LOG)
print("\nBring", OUT.name, "back to the repository for the Audit 0034 review.")
